In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

In [2]:
df = pd.read_csv('03-classification/course_lead_scoring_2026.csv')


In [3]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [5]:
categorical = list(df.dtypes[df.dtypes == 'str'].index)
numerical = list(df.dtypes[df.dtypes != 'str'].index)

In [6]:
for c in categorical:
    df[c]=df[c].fillna('NA')
for j in numerical:
    df[j]=df[j].fillna(0.0)

In [7]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

# Q1

In [8]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

# Q2

In [9]:
print(df[['interaction_count','number_of_courses_viewed']].corrwith(df.lead_score))
print(df[['number_of_courses_viewed','annual_income']].corrwith(df.interaction_count))

interaction_count           0.915746
number_of_courses_viewed    0.757204
dtype: float64
number_of_courses_viewed    0.721609
annual_income               0.122842
dtype: float64


In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [11]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [12]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

In [13]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

# Q3

In [14]:
categorical_subset =['industry','location','lead_source','employment_status']

In [15]:
def mutual_info_churn_score(series):
    return np.round(mutual_info_score(series, df_full_train.converted),2)

In [16]:
mi = df_full_train[categorical_subset].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64

# Q4

In [17]:
categorical = list(df.dtypes[df.dtypes == 'str'].index)
numerical = list(df.dtypes[df.dtypes != 'str'].index)
numerical.pop(-1)

'converted'

In [18]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [20]:
dv = DictVectorizer(sparse=False)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.fit_transform(val_dict)

y_pred = model.predict_proba(X_val)[:, 1]

In [21]:
churn_decision = (y_pred >= 0.5)

In [22]:
np.round((y_val == churn_decision).mean(),2)

np.float64(0.64)

# Q5

In [23]:
q4_accuracy = (y_val == churn_decision).mean()

In [24]:
categorical = list(df.dtypes[df.dtypes == 'str'].index)
numerical = list(df.dtypes[df.dtypes != 'str'].index)
numerical.pop(-1)
accuracy_list = []
lista_features=[]
columns= categorical + numerical

for c in columns:
    features_actuales = [col for col in columns if col != c]
    dv = DictVectorizer(sparse=False)
    
    train_dict = df_train[features_actuales].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    dv = DictVectorizer(sparse=False)
    val_dict = df_val[features_actuales].to_dict(orient='records')
    X_val = dv.fit_transform(val_dict)

    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    accuracy=(y_val == churn_decision).mean()
    accuracy_list.append(accuracy)
    lista_features.append(c)
    

In [25]:
df_results = pd.DataFrame({
    'deleted_feature': lista_features,
    'accuracy': accuracy_list
})

In [26]:
df_results['accuracy_dif']= np.abs(q4_accuracy-df_results['accuracy'])

In [27]:
df_results[df_results['deleted_feature'].isin(['lead_source','number_of_courses_viewed','interaction_count'])].sort_values(by='accuracy_dif',ascending=True)

,deleted_feature,accuracy,accuracy_dif
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


# Q6

In [28]:
categorical = list(df.dtypes[df.dtypes == 'str'].index)
numerical = list(df.dtypes[df.dtypes != 'str'].index)
numerical.pop(-1)
columns= categorical + numerical
accuracy_list = []
lista_c= []

C=[0.000001, 0.00001, 0.0001, 0.001]
for c in C:
    dv = DictVectorizer(sparse=False)
    
    train_dict = df_train[columns].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    dv = DictVectorizer(sparse=False)
    val_dict = df_val[columns].to_dict(orient='records')
    X_val = dv.fit_transform(val_dict)

    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    accuracy= np.round((y_val == churn_decision).mean(),3)
    accuracy_list.append(accuracy)
    lista_c.append(c)

In [29]:
df_results_2 = pd.DataFrame({
    'c_value': lista_c,
    'accuracy': accuracy_list
})

In [30]:
df_results_2.sort_values(by='accuracy',ascending=False)

,c_value,accuracy
3,0.001000,0.645
2,0.000100,0.613
1,0.000010,0.598
0,0.000001,0.598
